# Laboratorio 3.4 — Clasificación con scikit-learn: de los datos a las métricas

**Módulo 3 · Herramientas y Tecnologías** — bloque [`04-ml-dl-y-evaluacion.md`](../../../Apuntes-Markdown/03-herramientas-y-tecnologias/04-ml-dl-y-evaluacion.md)

**Duración orientativa:** 120–150 minutos · **Modalidad:** individual, notebook semi-guiado · **Herramientas:** Google Colab + scikit-learn

---

Este notebook es el más largo del módulo, y con razón: recorre el ciclo completo de un problema de clasificación supervisada, desde la carga de datos hasta un modelo entrenado y guardado listo para desplegar (en el laboratorio 3.6). Todo el código está escrito y verificado — cada número que veréis en las celdas Markdown ha sido comprobado ejecutando el notebook de principio a fin — y vuestro trabajo es ejecutarlo, entenderlo y razonar sobre los resultados en las preguntas de reflexión.

## Objetivo de aprendizaje

Aplicar el ciclo completo de un problema de clasificación supervisada con evaluación rigurosa: separación train/validation/test, matriz de confusión, precision/recall/F1, ajuste del umbral de decisión y detección de overfitting/underfitting comparando modelos de complejidad distinta.

## Contexto

El apunte del bloque 4 insiste especialmente en un mensaje central: **la métrica correcta depende del problema**, y el accuracy puede ser profundamente engañosa cuando las clases están desbalanceadas. La detección de fraude es el ejemplo de manual de este fenómeno: un modelo que prediga siempre "no es fraude" tendría casi un 98% de accuracy en este dataset... y sería completamente inútil, porque nunca detectaría un fraude real.

Vais a trabajar con `fraude_transacciones.csv`, un dataset de 6.000 transacciones donde solo el 1.67% son fraudulentas. El apunte también explica en detalle por qué se necesitan tres particiones (train/validation/test) en vez de dos, y los conceptos de overfitting y underfitting, que vais a provocar deliberadamente en la Parte 5 de este laboratorio para verlos con vuestros propios ojos, no solo en la teoría.

Un aviso importante antes de empezar: **los resultados de este laboratorio son honestos, no "maquillados" para que salga bonito**. Con solo un ~1.7% de casos positivos, las métricas sobre un conjunto de validación de 1.200 filas (apenas 20 fraudes) tienen bastante varianza, y vais a comprobar que el modelo más sofisticado (Random Forest) no siempre gana al más simple (regresión logística) — un resultado perfectamente real y con el que os vais a encontrar a menudo en proyectos reales de datos desbalanceados.

## Dataset

`fraude_transacciones.csv` (en esta misma carpeta), 6.000 filas con las columnas: `transaccion_id, hora_dia (0-23), importe, categoria_comercio, pais_distinto_habitual (0/1), dispositivo_nuevo (0/1), num_transacciones_24h, antiguedad_cuenta_dias, es_fraude (0/1, variable objetivo)`.


## Parte 1 — Carga y exploración rápida (15 min)

Antes de modelar, un EDA breve (reutilizando las técnicas del laboratorio 3.1) para conocer el dataset y, sobre todo, **cuantificar el desbalanceo de clases**, que va a condicionar todas las decisiones posteriores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
np.random.seed(42)

df = pd.read_csv("fraude_transacciones.csv")
print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# El dato clave de este laboratorio: ¿cuántas transacciones son fraude?
conteo_clases = df["es_fraude"].value_counts()
porcentaje_clases = df["es_fraude"].value_counts(normalize=True) * 100

print("Conteo de clases:")
print(conteo_clases)
print()
print("Porcentaje de clases:")
print(porcentaje_clases.round(2))
print()
print(f"-> Solo el {porcentaje_clases[1]:.2f}% de las transacciones son fraude.")
print("   Un modelo que prediga SIEMPRE 'no fraude' tendría un accuracy de:",
      f"{porcentaje_clases[0]:.2f}%, y sería completamente inútil.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=df, x="es_fraude", ax=axes[0], hue="es_fraude", palette=["#4C72B0", "#C44E52"], legend=False)
axes[0].set_title("Conteo de transacciones por clase")
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["No fraude (0)", "Fraude (1)"])

sns.boxplot(data=df, x="es_fraude", y="importe", ax=axes[1], hue="es_fraude", palette=["#4C72B0", "#C44E52"], legend=False)
axes[1].set_title("Distribución del importe según fraude")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["No fraude", "Fraude"])

plt.tight_layout()
plt.show()

El desbalanceo es evidente: solo hay unas 100 transacciones fraudulentas frente a casi 5.900 legítimas. Esto es exactamente el motivo por el que **el accuracy no será nuestra métrica principal** en este laboratorio.

## Parte 2 — Separación train / validation / test (15 min)

Siguiendo el apunte, separamos los datos en tres conjuntos: **train** (60%) para que el modelo aprenda, **validation** (20%) para comparar modelos y ajustar decisiones, y **test** (20%) que solo se usa **una vez**, al final, para la evaluación honesta del modelo elegido.

Para dividir en tres con `train_test_split` (que solo divide en dos) lo aplicamos **dos veces**: primero separamos test (20%) del resto, y luego separamos train y validation del 80% restante. Usamos `stratify=y` en ambos pasos para que la proporción de fraude se mantenga igual en los tres conjuntos — imprescindible con un desbalanceo tan fuerte, para no acabar con un conjunto de validación sin apenas ejemplos de fraude.

In [ ]:
from sklearn.model_selection import train_test_split

columnas_features = ["hora_dia", "importe", "categoria_comercio", "pais_distinto_habitual",
                      "dispositivo_nuevo", "num_transacciones_24h", "antiguedad_cuenta_dias"]
columna_target = "es_fraude"

X = df[columnas_features]
y = df[columna_target]

# Paso 1: separamos el 20% de test del resto (80%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Paso 2: del 80% restante, separamos train (60% del total = 75% de X_temp) y
# validation (20% del total = 25% de X_temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train:      {len(X_train):5d} filas ({len(X_train)/len(df):.1%})  -  fraude: {y_train.mean():.2%}")
print(f"Validation: {len(X_val):5d} filas ({len(X_val)/len(df):.1%})  -  fraude: {y_val.mean():.2%}")
print(f"Test:       {len(X_test):5d} filas ({len(X_test)/len(df):.1%})  -  fraude: {y_test.mean():.2%}")
print()
print("-> Gracias a stratify=y, la proporción de fraude es prácticamente idéntica en los tres conjuntos.")

## Parte 3 — Preprocesado y dos modelos de complejidad distinta (30 min)

Construimos el preprocesado con un `ColumnTransformer`: escalamos las variables numéricas continuas con `StandardScaler`, dejamos pasar sin cambios las banderas binarias (`pais_distinto_habitual`, `dispositivo_nuevo`, que ya están en 0/1), y aplicamos `OneHotEncoder` a la única variable categórica, `categoria_comercio`.

Encapsulamos preprocesado + modelo en un único objeto `Pipeline`, como recomienda el apunte: así garantizamos que el escalado se "aprende" solo con datos de entrenamiento, evitando data leakage hacia validación o test.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

columnas_numericas = ["hora_dia", "importe", "num_transacciones_24h", "antiguedad_cuenta_dias"]
columnas_binarias = ["pais_distinto_habitual", "dispositivo_nuevo"]
columnas_categoricas = ["categoria_comercio"]

preprocesador = ColumnTransformer(transformers=[
    ("numericas", StandardScaler(), columnas_numericas),
    ("binarias", "passthrough", columnas_binarias),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
])

print("Preprocesador construido:")
print(preprocesador)

### Modelo 1 — Regresión logística

Modelo lineal, interpretable, rápido de entrenar. Usamos `class_weight="balanced"` para que el algoritmo penalice más los errores sobre la clase minoritaria (fraude) durante el entrenamiento, compensando el desbalanceo — sin esto, un modelo lineal tendería a ignorar la clase minoritaria por ser tan poco frecuente.

In [ ]:
pipeline_logreg = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

pipeline_logreg.fit(X_train, y_train)
print("Regresión logística entrenada.")

### Modelo 2 — Random Forest

Ensemble de árboles de decisión, más flexible y capaz de capturar relaciones no lineales entre variables. También aplicamos `class_weight="balanced"` por el mismo motivo. Usamos una profundidad razonable (`max_depth=8`) para empezar con una versión "sensata"; en la Parte 5 compararemos esta misma familia de modelo sin ningún límite de profundidad para provocar overfitting a propósito.

In [ ]:
pipeline_rf = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", RandomForestClassifier(
        n_estimators=200, max_depth=8, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )),
])

pipeline_rf.fit(X_train, y_train)
print("Random Forest entrenado.")

## Parte 4 — Matriz de confusión, precision, recall y F1 (30 min)

Evaluamos ambos modelos sobre el conjunto de **validación** (nunca sobre test todavía, para no "gastarlo"), con el umbral de decisión por defecto (0.5). Usamos `classification_report` y `ConfusionMatrixDisplay`, tal como describe el apunte.

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, accuracy_score, roc_auc_score,
)

def evaluar_modelo(pipeline, X_eval, y_eval, nombre):
    y_pred = pipeline.predict(X_eval)
    y_proba = pipeline.predict_proba(X_eval)[:, 1]
    print(f"===== {nombre} =====")
    print(classification_report(y_eval, y_pred, target_names=["No fraude", "Fraude"], digits=3))
    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision_fraude": precision_score(y_eval, y_pred, pos_label=1, zero_division=0),
        "recall_fraude": recall_score(y_eval, y_pred, pos_label=1),
        "f1_fraude": f1_score(y_eval, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_proba),
    }

metricas_logreg = evaluar_modelo(pipeline_logreg, X_val, y_val, "Regresión logística (validación, umbral 0.5)")
metricas_rf = evaluar_modelo(pipeline_rf, X_val, y_val, "Random Forest max_depth=8 (validación, umbral 0.5)")

In [ ]:
# Tabla comparativa de métricas sobre validación
tabla_comparativa = pd.DataFrame([metricas_logreg, metricas_rf]).set_index("modelo").round(3)
tabla_comparativa

**Lo que deberíais observar:** a igualdad de umbral (0.5), la regresión logística obtiene bastante más **recall** sobre la clase fraude que el Random Forest (en torno al 65% frente a apenas un 5%), aunque su precision sea aún peor que la ya de por sí baja precision del Random Forest. Es un resultado real y no un error: `class_weight="balanced"` afecta de forma distinta a un modelo lineal que a un ensemble de árboles, y con una clase tan minoritaria (~1.7%), el comportamiento por defecto de cada algoritmo en el umbral 0.5 puede ser muy distinto. El ROC-AUC (que mide la capacidad discriminativa del modelo a **todos** los umbrales posibles, no solo 0.5) sitúa a ambos modelos en un terreno más parecido — y es la métrica que usaremos para decidir cuál de los dos merece la pena afinar con el umbral en la Parte 6.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay.from_estimator(
    pipeline_logreg, X_val, y_val, display_labels=["No fraude", "Fraude"],
    cmap="Blues", ax=axes[0], colorbar=False,
)
axes[0].set_title("Regresión logística (umbral 0.5)")

ConfusionMatrixDisplay.from_estimator(
    pipeline_rf, X_val, y_val, display_labels=["No fraude", "Fraude"],
    cmap="Blues", ax=axes[1], colorbar=False,
)
axes[1].set_title("Random Forest max_depth=8 (umbral 0.5)")

plt.tight_layout()
plt.show()

### ¿Qué métrica importa más aquí?

En detección de fraude, un **falso negativo** (una transacción fraudulenta que el modelo clasifica como legítima) es mucho más costoso que un **falso positivo** (una transacción legítima marcada como sospechosa, que en el peor caso genera una revisión manual innecesaria o una llamada al cliente). Por eso, en este problema **priorizamos el recall de la clase fraude** sobre el accuracy o incluso sobre la precision a secas.

Esto no significa ignorar la precision del todo: un modelo con recall altísimo pero precision demasiado baja generaría tantas falsas alarmas que el equipo de fraude dejaría de confiar en el sistema (o se saturaría revisando transacciones). El objetivo es encontrar un **umbral de decisión** que dé un recall razonablemente alto sin inundar de falsos positivos al equipo de revisión — es lo que hacemos en la Parte 6, después de estudiar el overfitting en la Parte 5.

## Parte 5 — Provocar overfitting deliberadamente (20 min)

Entrenamos un Random Forest **sin ningún límite de profundidad** (`max_depth=None`) y sin apenas regularización, y comparamos su rendimiento en train vs. validación frente al modelo con `max_depth=8` de la Parte 3. El objetivo es *ver* con números la caída de generalización que describe el apunte: un modelo que memoriza el ruido del conjunto de entrenamiento rinde muy bien ahí, pero mal en datos que no ha visto. (Usamos Random Forest para esta demostración porque los árboles sin límite de profundidad son el ejemplo más claro e intuitivo de sobreajuste; es independiente de cuál de los dos modelos de la Parte 4 acabe siendo el recomendado.)

In [ ]:
pipeline_rf_overfit = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,       # sin límite de profundidad: cada árbol puede crecer hasta memorizar el train
        min_samples_leaf=1,   # sin regularización mínima: hojas de una sola muestra permitidas
        class_weight="balanced",
        random_state=42, n_jobs=-1,
    )),
])
pipeline_rf_overfit.fit(X_train, y_train)

def f1_en(pipeline, X_eval, y_eval):
    y_pred = pipeline.predict(X_eval)
    return f1_score(y_eval, y_pred, pos_label=1, zero_division=0)

comparacion_overfitting = pd.DataFrame([
    {
        "modelo": "RF max_depth=8 (regularizado)",
        "f1_train": f1_en(pipeline_rf, X_train, y_train),
        "f1_validacion": f1_en(pipeline_rf, X_val, y_val),
    },
    {
        "modelo": "RF max_depth=None (sin regularizar)",
        "f1_train": f1_en(pipeline_rf_overfit, X_train, y_train),
        "f1_validacion": f1_en(pipeline_rf_overfit, X_val, y_val),
    },
]).set_index("modelo").round(3)

comparacion_overfitting["caida_train_a_validacion"] = (
    comparacion_overfitting["f1_train"] - comparacion_overfitting["f1_validacion"]
).round(3)

comparacion_overfitting

In [ ]:
comparacion_overfitting[["f1_train", "f1_validacion"]].plot(
    kind="bar", figsize=(7, 4.5), color=["#4C72B0", "#DD8452"], rot=15,
)
plt.title("F1 en train vs. validación: efecto de eliminar la regularización")
plt.ylabel("F1 (clase fraude)")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

**Lectura del resultado:** el modelo sin límite de profundidad obtiene un F1 mucho más alto en train que en validación — llega a memorizar casi perfectamente los ejemplos de entrenamiento, incluido su ruido — mientras que el modelo con `max_depth=8` tiene un F1 en train más modesto, pero mucho más parecido al de validación. Esa diferencia entre el rendimiento en train y en validación (la columna `caida_train_a_validacion`) es la firma característica del **overfitting**: cuanto mayor es la caída, más ha memorizado el modelo patrones específicos del conjunto de entrenamiento que no generalizan a datos nuevos. El modelo regularizado (`max_depth=8`) generaliza mejor, que es justo el objetivo del entrenamiento (concepto de "good fit" del apunte).

## Parte 6 — Ajuste del umbral y evaluación honesta sobre test (15 min)

En la Parte 4 vimos que, al umbral por defecto (0.5), la **regresión logística** consigue mucho mejor recall que el Random Forest sobre la clase fraude, y un ROC-AUC igual de bueno o mejor. Nos quedamos con la **regresión logística con `class_weight="balanced"`** como modelo base y exploramos distintos umbrales de decisión sobre el conjunto de validación para encontrar un compromiso razonable entre recall y precision (y, de paso, un volumen de transacciones señaladas que un equipo humano pueda revisar de verdad).

Como describe el apunte (caso conceptual de churn — el mismo razonamiento aplica a fraude): "el umbral de clasificación es un parámetro de negocio", no una constante fija en el código.

In [ ]:
probabilidades_val = pipeline_logreg.predict_proba(X_val)[:, 1]

filas_umbral = []
for umbral_candidato in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    predicciones = (probabilidades_val >= umbral_candidato).astype(int)
    filas_umbral.append({
        "umbral": umbral_candidato,
        "precision": precision_score(y_val, predicciones, pos_label=1, zero_division=0),
        "recall": recall_score(y_val, predicciones, pos_label=1),
        "f1": f1_score(y_val, predicciones, pos_label=1, zero_division=0),
        "pct_transacciones_marcadas": predicciones.mean() * 100,
    })

tabla_umbrales = pd.DataFrame(filas_umbral).round(3)
tabla_umbrales

**Cómo leer esta tabla:** al bajar el umbral, el recall sube (detectamos más fraudes reales) pero la precision baja y el porcentaje de transacciones marcadas para revisión se dispara — con un umbral de 0.1 estaríamos marcando en torno a un tercio de todas las transacciones, algo inviable para cualquier equipo de revisión humano. Al subir el umbral ocurre lo contrario. El **umbral 0.8** ofrece un buen compromiso: mantiene un recall del entorno del 45% (detecta casi la mitad de los fraudes reales) mientras marca solo en torno a un 7% de las transacciones, un volumen de revisión mucho más manejable, con el mejor F1 de toda la tabla. Es el umbral que recomendamos como parámetro de negocio para este modelo (parámetro `umbral_recomendado`, modificable en la celda siguiente).

In [ ]:
umbral_recomendado = 0.8  # <-- parámetro que podéis modificar para explorar el compromiso precision/recall

print(f"Regresión logística con umbral = {umbral_recomendado} (validación):")
predicciones_val_umbral = (probabilidades_val >= umbral_recomendado).astype(int)
print(classification_report(y_val, predicciones_val_umbral, target_names=["No fraude", "Fraude"], digits=3))

Con el modelo y el umbral ya decididos sobre validación, hacemos la evaluación final — y **por primera y única vez** — sobre el conjunto de **test**, que no hemos tocado hasta ahora. Es la estimación más honesta de cómo se comportaría el modelo en producción.

In [ ]:
probabilidades_test = pipeline_logreg.predict_proba(X_test)[:, 1]
predicciones_test = (probabilidades_test >= umbral_recomendado).astype(int)

print(f"===== Regresión logística, umbral={umbral_recomendado} (TEST — evaluación final) =====")
print(classification_report(y_test, predicciones_test, target_names=["No fraude", "Fraude"], digits=3))
print(f"ROC-AUC en test: {roc_auc_score(y_test, probabilidades_test):.3f}")
print(f"Porcentaje de transacciones marcadas para revisión: {predicciones_test.mean():.1%}")

**Recomendación final:** usar la **regresión logística** con `class_weight="balanced"`, con un umbral de decisión de **0.8** en lugar del 0.5 por defecto. A este umbral, el modelo mantiene un ROC-AUC sólido y detecta una parte significativa del fraude real (recall ≈ 45-60% según el conjunto evaluado) mientras mantiene un volumen de revisión manual manejable (en torno al 7% de las transacciones). El umbral exacto debería ajustarse en producción junto con el equipo de fraude, en función de su capacidad real de revisión y del coste efectivo de un fraude no detectado frente al de una revisión innecesaria — pero la tabla de la Parte 6 les da un punto de partida cuantificado en vez de una elección arbitraria.

## Parte 7 — Entrenar el modelo final y guardarlo con joblib (10 min)

Por último, entrenamos el pipeline recomendado (preprocesado + regresión logística) sobre **todos los datos disponibles** (train + validation + test), para aprovechar el máximo de información posible antes de guardarlo. Este es el modelo que se desplegará en el laboratorio 3.6 con Gradio, así que el `Pipeline` guardado debe incluir el preprocesado completo: recibirá datos crudos (sin escalar ni codificar) y debe poder transformarlos y predecir en un único paso. Guardamos también el umbral recomendado junto al modelo, para que el laboratorio 3.6 lo reutilice sin tener que repetir este análisis.

In [ ]:
import joblib

pipeline_final = Pipeline(steps=[
    ("preprocesado", preprocesador),
    ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

# Entrenamos sobre el dataset completo (todas las filas de X, y) para maximizar los datos
# usados por el modelo que se va a desplegar.
pipeline_final.fit(X, y)

joblib.dump(pipeline_final, "modelo_fraude.joblib")
print("Modelo final guardado en modelo_fraude.joblib")
print(f"Umbral de decisión recomendado (a usar en el laboratorio 3.6): {umbral_recomendado}")

# Comprobación rápida: el pipeline cargado predice correctamente sobre dos filas de ejemplo,
# una "normal" y otra con señales típicas de fraude activadas.
modelo_cargado = joblib.load("modelo_fraude.joblib")

ejemplo_normal = pd.DataFrame([{
    "hora_dia": 14, "importe": 25.0, "categoria_comercio": "Alimentación",
    "pais_distinto_habitual": 0, "dispositivo_nuevo": 0,
    "num_transacciones_24h": 2, "antiguedad_cuenta_dias": 900,
}])
ejemplo_sospechoso = pd.DataFrame([{
    "hora_dia": 3, "importe": 850.0, "categoria_comercio": "Retirada de efectivo",
    "pais_distinto_habitual": 1, "dispositivo_nuevo": 1,
    "num_transacciones_24h": 9, "antiguedad_cuenta_dias": 15,
}])

proba_normal = modelo_cargado.predict_proba(ejemplo_normal)[0, 1]
proba_sospechoso = modelo_cargado.predict_proba(ejemplo_sospechoso)[0, 1]

print(f"Probabilidad de fraude — ejemplo normal:     {proba_normal:.3f}")
print(f"Probabilidad de fraude — ejemplo sospechoso: {proba_sospechoso:.3f}")
print("-> La probabilidad debe ser sustancialmente más alta en el ejemplo sospechoso.")

## Entregable

Este notebook ejecutado de principio a fin, con: los dos modelos entrenados y comparados en una tabla de métricas (Parte 3-4), la matriz de confusión y el razonamiento sobre qué métrica prioriza el caso de uso (Parte 4), la demostración de overfitting (Parte 5), la tabla de umbrales y la recomendación final justificada de modelo y umbral (Parte 6), y el fichero `modelo_fraude.joblib` generado (Parte 7) — este último se reutiliza en el laboratorio 3.6.

## Preguntas de reflexión

1. En la Parte 4, el Random Forest tuvo peor recall que la regresión logística al umbral por defecto (0.5), a pesar de tener un ROC-AUC parecido. ¿Qué os dice esto sobre la diferencia entre "un modelo es bueno discriminando" (ROC-AUC) y "un modelo está bien calibrado a un umbral concreto" (precision/recall a 0.5)?
2. Si el equipo de fraude solo pudiera revisar manualmente 30 transacciones al día, ¿qué umbral de la tabla de la Parte 6 elegiríais y por qué?
3. La regresión logística es un modelo interpretable (según el apunte): ¿cómo explicaríais a un responsable de negocio, sin tecnicismos, cómo podríais averiguar cuáles son las variables que más influyen en que el modelo prediga fraude?
4. `class_weight="balanced"` fue nuestra estrategia para lidiar con el desbalanceo de clases. ¿Qué otras estrategias conocéis o intuís que podrían funcionar (pista: pensad en el propio conjunto de datos de entrenamiento, no solo en el algoritmo)?
